In [2]:
import sys
sys.path.append("../src/")  # Agrega el directorio src al path

In [3]:
import os
DOCUMENTS_PATH = os.path.abspath("../data/raw/")
DOCUMENTS_PATH

'/Users/aingeru/workspace/Máster - TFM/data/raw'

In [4]:
pdfs = [file for file in os.listdir(DOCUMENTS_PATH) if file.endswith(".pdf") or file.endswith(".PDF")]

print(f"Número de documentos en la carpeta '{DOCUMENTS_PATH}': {len(pdfs)}")

Número de documentos en la carpeta '/Users/aingeru/workspace/Máster - TFM/data/raw': 42


Choose parser and chunker:

In [5]:
from parser import PDFParser
from chunking import PDFChunker

parser = PDFParser()
chunker = PDFChunker()

/Users/aingeru/workspace/Máster - TFM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iterate over all docs:

In [6]:
from tqdm import tqdm

chunks = []
for pdf in tqdm(pdfs, desc="Parseing and chunking PDFs"):
    file_path = os.path.join(DOCUMENTS_PATH, pdf)

    # Parsear el PDF
    file_documents = parser.parse(file_path)
    
    # Chunking de los documentos
    file_chunks = chunker.chunk(file_documents)

    chunks.extend(file_chunks)
    
print(f"Número total de chunks generados para todos los documentos: {len(chunks)}")

Parseing and chunking PDFs: 100%|██████████| 43/43 [01:37<00:00,  2.27s/it]

Número total de chunks generados para todos los documentos: 16200


In [8]:
display(type(chunks[0]))
display(chunks[0].metadata)
display(chunks[0].page_content[0:200])

display(chunks[5].metadata)
display(chunks[5].page_content[0:200])

langchain_core.documents.base.Document

{'producer': 'Antenna House PDF Output Library 6.6.1477 (Linux64)',
 'creator': 'AH XSL Formatter V6.6 MR7 for Linux64 : 6.6.9.39847 (2019-07-29T09:58+09)',
 'creationdate': '2025-09-16T12:16:26+01:00',
 'title': 'Ley 16/2003, de 28 de mayo, de cohesión y calidad del Sistema Nacional de Salud.',
 'author': 'Agencia Estatal Boletín Oficial del Estado',
 'subject': 'BOE-A-2003-10715 actualizado a 31 de octubre de 2024',
 'keywords': 'BOE-A-2003-10715; BOE; Legislación consolidada; Agencia Estatal Boletín Oficial del Estado',
 'moddate': '2025-09-16T12:16:26+01:00',
 'trapped': '/False',
 'source': '/Users/aingeru/workspace/Máster - TFM/data/raw/BOE-A-2003-10715-consolidado.pdf',
 'total_pages': 47,
 'page': 0,
 'page_label': '1'}

'Ley 16/2003, de 28 de mayo, de cohesión y calidad del Sistema \nNacional de Salud.\nJefatura del Estado\n«BOE» núm. 128, de 29 de mayo de 2003\nReferencia: BOE-A-2003-10715\nÍNDICE\n   \nPreámbulo ..........'

{'producer': 'Antenna House PDF Output Library 6.6.1477 (Linux64)',
 'creator': 'AH XSL Formatter V6.6 MR7 for Linux64 : 6.6.9.39847 (2019-07-29T09:58+09)',
 'creationdate': '2025-09-16T12:16:26+01:00',
 'title': 'Ley 16/2003, de 28 de mayo, de cohesión y calidad del Sistema Nacional de Salud.',
 'author': 'Agencia Estatal Boletín Oficial del Estado',
 'subject': 'BOE-A-2003-10715 actualizado a 31 de octubre de 2024',
 'keywords': 'BOE-A-2003-10715; BOE; Legislación consolidada; Agencia Estatal Boletín Oficial del Estado',
 'moddate': '2025-09-16T12:16:26+01:00',
 'trapped': '/False',
 'source': '/Users/aingeru/workspace/Máster - TFM/data/raw/BOE-A-2003-10715-consolidado.pdf',
 'total_pages': 47,
 'page': 1,
 'page_label': '2'}

'mpo. ................................................. 23\nArtículo 26. Garantías de información. .............................................. 23\nArtículo 27. Garantías de seguridad. ................'

Ingest documents:

In [ ]:
from time import time
from embedding import HuggingFaceEmbeddingsLC

embedder = HuggingFaceEmbeddingsLC()

# ingest all chunks and preserve metadata
start_time = time()
embeddings = embedder.embed_documents([chunk.page_content for chunk in chunks])
end_time = time()
print(f"Tiempo total para generar embeddings: {end_time - start_time:.2f} segundos")

# Combine embeddings with metadata
embedded_chunks = []
for chunk, embedding in zip(chunks, embeddings):
    embedded_chunk = {
        "embedding": embedding,
        "metadata": chunk.metadata,
        "page_content": chunk.page_content
    }
    embedded_chunks.append(embedded_chunk)

MPS is available. Using Apple Silicon GPU for embeddings.


Batches:   3%|▎         | 16/507 [00:43<24:40,  3.01s/it]